In [1]:
%%writefile generate_ct_images.py
"""
╔══════════════════════════════════════════════════════════╗
║  Diffusion Model Inference — Generate Synthetic CT Scans ║
║  Loads a trained checkpoint and generates images          ║
╚══════════════════════════════════════════════════════════╝

Usage (Kaggle or local):
    python generate_ct_images.py \
        --checkpoint /path/to/best_diffusion_model.pt \
        --output ./generated_images \
        --num_benign 300 \
        --num_malignant 0 \
        --num_normal 100 \
        --cfg_scale 3.0 \
        --ddim_steps 50 \
        --device cuda

Or in a Kaggle notebook cell:
    !python generate_ct_images.py \
        --checkpoint /kaggle/input/your-model/best_diffusion_model.pt \
        --output /kaggle/working/synthetic_images
"""

import argparse
import copy
import math
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.utils import save_image, make_grid
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


# ═══════════════════════════════════════════════════════════════════════
# MODEL ARCHITECTURE (must match training exactly)
# ═══════════════════════════════════════════════════════════════════════

class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        q = self.q(h).view(B, C, H * W)
        k = self.k(h).view(B, C, H * W)
        v = self.v(h).view(B, C, H * W)
        attn = torch.bmm(q.permute(0, 2, 1), k) * (C ** -0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(B, C, H, W)
        return x + self.proj(out)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, class_emb_dim, use_attention=False):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.GroupNorm(8, in_ch), nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        )
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch))
        self.class_mlp = nn.Sequential(nn.SiLU(), nn.Linear(class_emb_dim, out_ch))
        self.conv2 = nn.Sequential(
            nn.GroupNorm(8, out_ch), nn.SiLU(), nn.Dropout(0.1),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.attention = SelfAttention(out_ch) if use_attention else nn.Identity()

    def forward(self, x, t_emb, c_emb):
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = h + self.class_mlp(c_emb)[:, :, None, None]
        h = self.conv2(h)
        h = h + self.skip(x)
        h = self.attention(h)
        return h


class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_channels=64,
                 time_emb_dim=256, num_classes=3, class_emb_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim), nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.class_embed = nn.Embedding(num_classes + 1, class_emb_dim)
        ch = base_channels

        # Encoder
        self.enc_conv0 = nn.Conv2d(img_channels, ch, 3, padding=1)
        self.enc1 = ResBlock(ch, ch, time_emb_dim, class_emb_dim)
        self.down1 = nn.Conv2d(ch, ch, 4, stride=2, padding=1)
        self.enc2 = ResBlock(ch, ch * 2, time_emb_dim, class_emb_dim)
        self.down2 = nn.Conv2d(ch * 2, ch * 2, 4, stride=2, padding=1)
        self.enc3 = ResBlock(ch * 2, ch * 4, time_emb_dim, class_emb_dim, use_attention=True)
        self.down3 = nn.Conv2d(ch * 4, ch * 4, 4, stride=2, padding=1)

        # Bottleneck
        self.bot1 = ResBlock(ch * 4, ch * 8, time_emb_dim, class_emb_dim, use_attention=True)
        self.bot2 = ResBlock(ch * 8, ch * 8, time_emb_dim, class_emb_dim, use_attention=True)
        self.bot3 = ResBlock(ch * 8, ch * 4, time_emb_dim, class_emb_dim)

        # Decoder
        self.up3 = nn.ConvTranspose2d(ch * 4, ch * 4, 4, stride=2, padding=1)
        self.dec3 = ResBlock(ch * 8, ch * 2, time_emb_dim, class_emb_dim, use_attention=True)
        self.up2 = nn.ConvTranspose2d(ch * 2, ch * 2, 4, stride=2, padding=1)
        self.dec2 = ResBlock(ch * 4, ch, time_emb_dim, class_emb_dim)
        self.up1 = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)
        self.dec1 = ResBlock(ch * 2, ch, time_emb_dim, class_emb_dim)

        # Output
        self.out = nn.Sequential(
            nn.GroupNorm(8, ch), nn.SiLU(),
            nn.Conv2d(ch, img_channels, 3, padding=1),
        )

    def forward(self, x, t, c):
        t_emb = self.time_embed(t)
        c_emb = self.class_embed(c)

        x0 = self.enc_conv0(x)
        e1 = self.enc1(x0, t_emb, c_emb)
        e1_down = self.down1(e1)
        e2 = self.enc2(e1_down, t_emb, c_emb)
        e2_down = self.down2(e2)
        e3 = self.enc3(e2_down, t_emb, c_emb)
        e3_down = self.down3(e3)

        b = self.bot1(e3_down, t_emb, c_emb)
        b = self.bot2(b, t_emb, c_emb)
        b = self.bot3(b, t_emb, c_emb)

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3, t_emb, c_emb)
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2, t_emb, c_emb)
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1, t_emb, c_emb)

        return self.out(d1)


# ═══════════════════════════════════════════════════════════════════════
# DDIM SAMPLER
# ═══════════════════════════════════════════════════════════════════════

class DiffusionSampler:
    """Lightweight sampler for inference — only needs DDIM, no training logic."""

    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02,
                 num_classes=3, device='cuda'):
        self.timesteps = timesteps
        self.num_classes = num_classes
        self.device = device

        betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        alphas = 1.0 - betas
        self.alphas_cumprod = torch.cumprod(alphas, dim=0)

    @torch.no_grad()
    def sample_ddim(self, model, shape, class_labels,
                    num_steps=50, cfg_scale=3.0, eta=0.0):
        """
        DDIM sampling with Classifier-Free Guidance.

        Args:
            model: Trained ConditionalUNet
            shape: (B, C, H, W) output shape
            class_labels: (B,) class indices (0=Benign, 1=Malignant, 2=Normal)
            num_steps: DDIM steps (50 = fast, 100 = higher quality)
            cfg_scale: Guidance strength (3.0 = balanced, 5.0+ = stronger class)
            eta: Stochasticity (0 = deterministic DDIM, 1 = full DDPM)
        """
        model.eval()
        b = shape[0]
        img = torch.randn(shape, device=self.device)

        # Sub-sequence of timesteps
        step_size = self.timesteps // num_steps
        timesteps = list(reversed(range(0, self.timesteps, step_size)))

        for i, t in enumerate(tqdm(timesteps, desc="DDIM Sampling")):
            t_batch = torch.full((b,), t, device=self.device, dtype=torch.long)

            # Classifier-Free Guidance
            noise_cond = model(img, t_batch, class_labels)
            noise_uncond = model(img, t_batch,
                                 torch.full_like(class_labels, self.num_classes))
            noise_pred = noise_uncond + cfg_scale * (noise_cond - noise_uncond)

            # DDIM update
            alpha_t = self.alphas_cumprod[t]
            if i + 1 < len(timesteps):
                alpha_prev = self.alphas_cumprod[timesteps[i + 1]]
            else:
                alpha_prev = torch.tensor(1.0, device=self.device)

            pred_x0 = (img - torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            sigma = eta * torch.sqrt(
                (1 - alpha_prev) / (1 - alpha_t) * (1 - alpha_t / alpha_prev)
            )
            dir_xt = torch.sqrt(1 - alpha_prev - sigma**2) * noise_pred

            if sigma > 0 and i + 1 < len(timesteps):
                noise = torch.randn_like(img)
                img = torch.sqrt(alpha_prev) * pred_x0 + dir_xt + sigma * noise
            else:
                img = torch.sqrt(alpha_prev) * pred_x0 + dir_xt

        return torch.clamp(img, -1.0, 1.0)


# ═══════════════════════════════════════════════════════════════════════
# INFERENCE FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════

CLASS_NAMES = {0: 'Benign', 1: 'Malignant', 2: 'Normal'}


def load_model(checkpoint_path, device='cuda'):
    """Load the trained model from a checkpoint."""
    print(f"\n📦 Loading checkpoint: {checkpoint_path}")

    model = ConditionalUNet(
        img_channels=1,
        base_channels=64,
        time_emb_dim=256,
        num_classes=3,
        class_emb_dim=256,
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Try EMA weights first (better quality), fall back to model weights
    if 'ema_state_dict' in ckpt:
        model.load_state_dict(ckpt['ema_state_dict'])
        print("  ✅ Loaded EMA weights (best quality)")
    elif 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
        print("  ✅ Loaded model weights")
    else:
        # Maybe the checkpoint IS the state dict
        model.load_state_dict(ckpt)
        print("  ✅ Loaded state dict directly")

    epoch = ckpt.get('epoch', '?')
    loss = ckpt.get('loss', ckpt.get('best_loss', '?'))
    print(f"  📍 Checkpoint epoch: {epoch} | Loss: {loss}")

    model.eval()
    return model


def generate_images(model, sampler, class_idx, num_images,
                    output_dir, batch_size=16, cfg_scale=3.0,
                    ddim_steps=50, device='cuda'):
    """Generate and save synthetic images for a specific class."""

    class_name = CLASS_NAMES[class_idx]
    save_dir = os.path.join(output_dir, class_name)
    os.makedirs(save_dir, exist_ok=True)

    print(f"\n🎨 Generating {num_images} synthetic {class_name} images...")

    generated = 0
    while generated < num_images:
        current_batch = min(batch_size, num_images - generated)
        labels = torch.full((current_batch,), class_idx,
                            device=device, dtype=torch.long)

        images = sampler.sample_ddim(
            model,
            shape=(current_batch, 1, 128, 128),
            class_labels=labels,
            num_steps=ddim_steps,
            cfg_scale=cfg_scale,
        )

        # Save individual images
        for i in range(current_batch):
            img = (images[i] + 1) / 2  # [-1,1] → [0,1]
            img = img.clamp(0, 1)
            path = os.path.join(save_dir,
                                f"synthetic_{class_name}_{generated + i:04d}.png")
            save_image(img, path)

        generated += current_batch
        print(f"  💾 Saved {generated}/{num_images}")

    print(f"  ✅ Done! Images saved to {save_dir}")
    return save_dir


def generate_preview_grid(model, sampler, output_dir,
                          cfg_scale=3.0, ddim_steps=50, device='cuda'):
    """Generate a 3×8 preview grid (one row per class)."""

    print("\n🖼️  Generating preview grid...")
    fig, axes = plt.subplots(3, 8, figsize=(20, 8))
    fig.suptitle('Synthetic CT Samples\n(Rows: Benign, Malignant, Normal)',
                 fontsize=14, fontweight='bold')

    for class_idx in range(3):
        labels = torch.full((8,), class_idx, device=device, dtype=torch.long)
        samples = sampler.sample_ddim(
            model,
            shape=(8, 1, 128, 128),
            class_labels=labels,
            num_steps=ddim_steps,
            cfg_scale=cfg_scale,
        )

        for j in range(8):
            img = (samples[j, 0].cpu() + 1) / 2
            axes[class_idx, j].imshow(img, cmap='gray')
            axes[class_idx, j].axis('off')
            if j == 0:
                axes[class_idx, j].set_ylabel(CLASS_NAMES[class_idx], fontsize=12)

    plt.tight_layout()
    grid_path = os.path.join(output_dir, 'synthetic_preview_grid.png')
    plt.savefig(grid_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ Grid saved to {grid_path}")
    return grid_path


def generate_cfg_comparison(model, sampler, output_dir,
                            ddim_steps=50, device='cuda'):
    """Generate comparison at different CFG scales to find best quality."""

    print("\n📊 Generating CFG scale comparison...")
    cfg_scales = [1.0, 2.0, 3.0, 5.0, 7.0]
    fig, axes = plt.subplots(len(cfg_scales), 6, figsize=(15, len(cfg_scales) * 2.5))
    fig.suptitle('CFG Scale Comparison (Higher = Stronger Class Conditioning)',
                 fontsize=13, fontweight='bold')

    # Fixed seed for fair comparison
    seed = torch.manual_seed(42)

    for row, scale in enumerate(cfg_scales):
        torch.manual_seed(42)  # Same noise for each row
        for class_idx in range(3):
            labels = torch.full((2,), class_idx, device=device, dtype=torch.long)
            samples = sampler.sample_ddim(
                model,
                shape=(2, 1, 128, 128),
                class_labels=labels,
                num_steps=ddim_steps,
                cfg_scale=scale,
            )
            for k in range(2):
                col = class_idx * 2 + k
                img = (samples[k, 0].cpu() + 1) / 2
                axes[row, col].imshow(img, cmap='gray')
                axes[row, col].axis('off')
                if col == 0:
                    axes[row, col].set_ylabel(f'CFG={scale}', fontsize=10)
                if row == 0:
                    axes[row, col].set_title(
                        f'{CLASS_NAMES[class_idx]}', fontsize=9)

    plt.tight_layout()
    path = os.path.join(output_dir, 'cfg_comparison.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ CFG comparison saved to {path}")


# ═══════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════

def main():
    parser = argparse.ArgumentParser(
        description='Generate synthetic CT images from trained diffusion model')

    parser.add_argument('--checkpoint', type=str, required=True,
                        help='Path to checkpoint (.pt file)')
    parser.add_argument('--output', type=str, default='./generated_images',
                        help='Output directory')
    parser.add_argument('--num_benign', type=int, default=300,
                        help='Number of Benign images to generate')
    parser.add_argument('--num_malignant', type=int, default=0,
                        help='Number of Malignant images to generate')
    parser.add_argument('--num_normal', type=int, default=100,
                        help='Number of Normal images to generate')
    parser.add_argument('--cfg_scale', type=float, default=3.0,
                        help='Classifier-Free Guidance scale (3.0=balanced, 5.0+=stronger)')
    parser.add_argument('--ddim_steps', type=int, default=50,
                        help='DDIM sampling steps (50=fast, 100=quality)')
    parser.add_argument('--batch_size', type=int, default=16,
                        help='Generation batch size')
    parser.add_argument('--device', type=str, default='cuda',
                        help='Device (cuda or cpu)')
    parser.add_argument('--preview_only', action='store_true',
                        help='Only generate preview grid, no full batch')
    parser.add_argument('--cfg_compare', action='store_true',
                        help='Generate CFG scale comparison chart')
    parser.add_argument('--seed', type=int, default=42,
                        help='Random seed for reproducibility')

    args = parser.parse_args()

    # Setup
    device = args.device
    if device == 'cuda' and not torch.cuda.is_available():
        print("⚠️  CUDA not available, falling back to CPU")
        device = 'cpu'

    torch.manual_seed(args.seed)
    os.makedirs(args.output, exist_ok=True)

    print("=" * 60)
    print("  🧬 Diffusion Model CT Image Generator")
    print("=" * 60)
    print(f"  Device:      {device}")
    print(f"  Checkpoint:  {args.checkpoint}")
    print(f"  Output:      {args.output}")
    print(f"  CFG Scale:   {args.cfg_scale}")
    print(f"  DDIM Steps:  {args.ddim_steps}")
    print(f"  Seed:        {args.seed}")

    # Load model
    model = load_model(args.checkpoint, device)

    # Create sampler
    sampler = DiffusionSampler(
        timesteps=1000, beta_start=1e-4, beta_end=0.02,
        num_classes=3, device=device,
    )

    # Always generate preview grid
    generate_preview_grid(model, sampler, args.output,
                          cfg_scale=args.cfg_scale,
                          ddim_steps=args.ddim_steps, device=device)

    # Optional: CFG comparison
    if args.cfg_compare:
        generate_cfg_comparison(model, sampler, args.output,
                                ddim_steps=args.ddim_steps, device=device)

    # Generate full batches (skip if preview_only)
    if not args.preview_only:
        plan = {
            0: args.num_benign,
            1: args.num_malignant,
            2: args.num_normal,
        }

        for class_idx, num in plan.items():
            if num > 0:
                generate_images(
                    model, sampler, class_idx, num, args.output,
                    batch_size=args.batch_size, cfg_scale=args.cfg_scale,
                    ddim_steps=args.ddim_steps, device=device,
                )

    # Summary
    print("\n" + "=" * 60)
    print("  ✅ GENERATION COMPLETE!")
    print("=" * 60)
    total = args.num_benign + args.num_malignant + args.num_normal
    if not args.preview_only:
        print(f"  Total images: {total}")
        for cls_idx, num in {0: args.num_benign, 1: args.num_malignant,
                              2: args.num_normal}.items():
            if num > 0:
                print(f"    {CLASS_NAMES[cls_idx]}: {num}")
    print(f"  Output dir:   {args.output}")
    print(f"  Preview grid: {args.output}/synthetic_preview_grid.png")


if __name__ == '__main__':
    main()


Writing generate_ct_images.py


In [2]:
!python generate_ct_images.py \
    --checkpoint /kaggle/input/models/tahmidulkashfi/diff-aug-3000epoch/pytorch/default/1/latest_checkpoint.pt \
    --output /kaggle/working/synthetic_images

  🧬 Diffusion Model CT Image Generator
  Device:      cuda
  Checkpoint:  /kaggle/input/models/tahmidulkashfi/diff-aug-3000epoch/pytorch/default/1/latest_checkpoint.pt
  Output:      /kaggle/working/synthetic_images
  CFG Scale:   3.0
  DDIM Steps:  50
  Seed:        42

📦 Loading checkpoint: /kaggle/input/models/tahmidulkashfi/diff-aug-3000epoch/pytorch/default/1/latest_checkpoint.pt
  ✅ Loaded EMA weights (best quality)
  📍 Checkpoint epoch: 3000 | Loss: 0.056208047084510326

🖼️  Generating preview grid...
DDIM Sampling: 100%|████████████████████████████| 50/50 [00:05<00:00,  8.62it/s]
  ✅ Grid saved to /kaggle/working/synthetic_images/synthetic_preview_grid.png

🎨 Generating 300 synthetic Benign images...
DDIM Sampling: 100%|████████████████████████████| 50/50 [00:10<00:00,  4.73it/s]
  💾 Saved 16/300
DDIM Sampling: 100%|████████████████████████████| 50/50 [00:10<00:00,  4.70it/s]
  💾 Saved 32/300
DDIM Sampling: 100%|████████████████████████████| 50/50 [00:10<00:00,  4.65it/s]
  💾 S

In [3]:
import shutil

shutil.make_archive("synthetic_images", "zip", "/kaggle/working/synthetic_images")


'/kaggle/working/synthetic_images.zip'